# RL Exper

In [1]:
# !pip install stable-baselines3 --default-timeout=100

In [ ]:
# train_uniswap_simple.py
# -----------------------
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ───── your environment ─────
from env_nikolai3 import UniswapV3LPGymEnv
from config.env_config import Config

# ───── RL + utils ─────
import gymnasium as gym
from gymnasium.wrappers import TimeLimit, RecordEpisodeStatistics, FlattenObservation
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy
import matplotlib.pyplot as plt
import pandas as pd
import datetime as dt

# ───── hyper‑params you might tweak ─────
TOTAL_TIMESTEPS      = 10_000     # training steps
MAX_EPISODE_MINUTES  = 1_000     # one‑day cap

# ╭────────────────────────── helpers ───────────────────────────╮
def make_env():
    cfg = Config()
    env = UniswapV3LPGymEnv(cfg, feat_num=19)

    env = TimeLimit(env, max_episode_steps=MAX_EPISODE_MINUTES)
    env = RecordEpisodeStatistics(env)
    env = FlattenObservation(env)

    logs_dir = Path("logs");  logs_dir.mkdir(exist_ok=True)
    env = Monitor(env, str(logs_dir), allow_early_resets=True)   # ← str() fix
    return env


def plot_learning_curve(monitor_csv: Path, title: str = "Learning curve") -> None:
    df = pd.read_csv(monitor_csv, comment="#")
    df["rolling_return"] = df["r"].rolling(window=20).mean()
    plt.figure(figsize=(8, 4))
    plt.plot(df["l"], df["rolling_return"])
    plt.xlabel("Episode"); plt.ylabel("Reward (20‑ep MA)")
    plt.title(title); plt.grid(True)
    plt.tight_layout(); plt.show()


# ╭────────────────────────── training ──────────────────────────╮
def main():
    # 1⃣  make vectorised env
    env = DummyVecEnv([make_env])

    # 2⃣  define agent
    model = PPO(
        "MlpPolicy",
        env,
        # n_steps       = 2048,
        n_steps       = 1024,
        batch_size    = 256,
        gae_lambda    = 0.95,
        gamma         = 0.999,
        learning_rate = 3e-4,
        clip_range    = 0.2,
        vf_coef       = 0.5,
        ent_coef      = 0.0,
        verbose       = 1,
        # tensorboard_log="tensorboard",
        device="auto",
    )

    # 3⃣  learn
    model.learn(total_timesteps=TOTAL_TIMESTEPS, progress_bar=True)

    # 4⃣  save & evaluate
    ts = dt.datetime.now().strftime("%Y%m%d_%H%M")
    model_path = Path("models"); model_path.mkdir(exist_ok=True)
    fname = model_path / f"ppo_uniswap_{ts}"
    model.save(fname)
    print(f"\n✔️  Saved model to {fname}.zip")

    mean_r, std_r = evaluate_policy(model, env, n_eval_episodes=10)
    print(f"Evaluation over 10 eps → mean ± std reward: {mean_r:.2f} ± {std_r:.2f}")

    # 5⃣  plot learning curve
    monitors = sorted(Path("logs").glob("*.monitor.csv"))
    if monitors:
        plot_learning_curve(monitors[-1], title="PPO on Uniswap‑V3 LP env")
    else:
        print("No monitor file found ‑ nothing to plot.")


if __name__ == "__main__":  
    main()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path
import torch


def tick_to_price(tick: int) -> float:
    sqrtp = 1.0001 ** (tick / 2)
    return sqrtp ** 2


def plot_learning_curve(log_dir: Path):
    """Plot episodic returns + 20-episode moving average."""
    mon_csv = sorted(log_dir.glob("monitor.csv"))[-1]
    df = pd.read_csv(mon_csv, comment="#")
    ep_returns = df["r"].to_numpy()
    ma = pd.Series(ep_returns).rolling(20).mean()

    plt.figure(figsize=(7, 4))
    plt.plot(ep_returns, label="episode return", alpha=0.6)
    plt.plot(ma, label="20-ep MA", linewidth=2)
    plt.xlabel("Episode")
    plt.ylabel("Reward")
    plt.title("Training curve – PPO on Uniswap-V3 LP env")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


def run_episode_and_plot(model, env):
    """Roll one episode, record price / actions, then plot & print actions."""
    obs, _ = env.reset()

    prices, plow, phigh = [], [], []
    mints_x, mints_y, burns_x, burns_y = [], [], [], []
    cum_pnl = []
    actions_out = []

    done = False
    step = 0
    while not done and step < env.EPISODE_LEN:
        ts = env.decision_grid[env.idx]
        p = env._eth_price(ts)

        # policy action
        action, _ = model.predict(obs, deterministic=True)
        actions_out.append(action.copy())

        pre_active = env.active           # ← state BEFORE step
        obs, reward, done, trunc, info = env.step(action)
        post_active = env.active          # ← state AFTER step

        prices.append(p)
        if post_active:
            plow.append(tick_to_price(env.tick_l))
            phigh.append(tick_to_price(env.tick_u))
        else:
            plow.append(np.nan)
            phigh.append(np.nan)

        # Mint event: inactive → active
        if (pre_active is False) and (post_active is True):
            mints_x.append(step)
            mints_y.append(p)
        # Burn event: active → inactive
        elif (pre_active is True) and (post_active is False):
            burns_x.append(step)
            burns_y.append(p)

        cum_pnl.append(env.cumulative_pnl)
        step += 1

    t = np.arange(len(prices))

    print(plow)
    print(phigh)

    fig, axs = plt.subplots(2, 1, figsize=(10, 6), sharex=True,
                            gridspec_kw={"height_ratios": [3, 1]})

    # (a) price + agent band
    axs[0].plot(t, prices, label="ETH-USDC price", linewidth=1.2)
    axs[0].fill_between(t, plow, phigh, color="tab:cyan", alpha=0.25,
                        label="agent tick-range")
    axs[0].scatter(mints_x, mints_y, marker="^", color="green", label="Mint")
    axs[0].scatter(burns_x, burns_y, marker="v", color="red", label="Burn")
    axs[0].set_ylabel("Price (USDC)")
    axs[0].set_title("Agent behaviour in one episode")
    axs[0].legend(loc="upper left")
    axs[0].grid(True)

    # (b) cumulative PnL
    axs[1].plot(t, cum_pnl, label="cumulative PnL")
    axs[1].set_ylabel("PnL (USDC)")
    axs[1].set_xlabel("Step (minute)")
    axs[1].grid(True)

    plt.tight_layout()
    plt.show()

    # show the raw 2-D actions
    actions_arr = np.array(actions_out)
    print(f"Raw actions from policy (shape {actions_arr.shape}):")
    np.set_printoptions(precision=3, suppress=True)
    print(actions_arr)


if __name__ == "__main__":
    LOG_DIR = Path("logs")
    MODEL_DIR = Path("models")

    plot_learning_curve(LOG_DIR)

    latest_model = sorted(MODEL_DIR.glob("ppo_uniswap_*.zip"))[-1]
    from stable_baselines3 import PPO
    from env_nikolai3 import UniswapV3LPGymEnv
    from config.env_config import Config

    env = UniswapV3LPGymEnv(Config(), feat_num=19)
    env.EPISODE_LEN = 1000  # IMPORTANT

    model = PPO.load(latest_model, env=env)
    run_episode_and_plot(model, env)